In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
import tensorflow as tf

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth activado en GPU")
    except RuntimeError as e:
        print(e)

I0000 00:00:1779209124.752871 1413974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779209124.807026 1413974 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779209126.136870 1413974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Memory growth activado en GPU


In [3]:
import random
import time
import joblib
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
# from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ART

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier, SklearnClassifier, XGBoostClassifier, LightGBMClassifier, CatBoostARTClassifier, KerasClassifier

/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/art/estimators/certification/__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


In [4]:
# Configuración de visualización

sns.set_style("whitegrid")

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
MLP_ATTACK_DEVICE = TRAIN_DEVICE
CNN_ATTACK_DEVICE = "/CPU:0"
INFER_DEVICE = "/CPU:0"
ATTACK_BATCH_SIZE = 128

SEED = 42
MODEL_DIR = os.path.join("model", "iot-23")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

if HAS_GPU:
    print("GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.")
else:
    print("No hay GPU disponible. Todo el notebook se ejecutará en CPU.")

tf.keras.backend.clear_session()

def build_mlp_model(input_dim, hidden_units):
    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape=(input_dim,)))
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.


In [5]:
# ==========================================
# 1. CARGA DE DATOS
# ==========================================

TARGET_COL = "label"
prepared_path = "../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv"

df_encoded = pl.read_csv(prepared_path)

feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy().astype(np.float32)

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np[train_full_idx],
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]
y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]

X_train_np = X_np[train_idx]
X_val_np = X_np[val_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X_np):,}")
print("La clase 0 corresponde a Benign y la clase 1 a Malicious.")


Entrenamiento: 745,504 muestras
Validación:    186,376 muestras
Test:          232,971 muestras
Clases en test: [0 1]
Total muestras: 1,164,851
La clase 0 corresponde a Benign y la clase 1 a Malicious.


In [6]:
# ==========================================
# 2. CONFIGURACIÓN DE LOS GANADORES
# ==========================================

RF_CONFIG = {"n_estimators": 50, "max_depth": 21}
XGB_CONFIG = {"n_estimators": 50, "max_depth": 6, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 50, "num_leaves": 15, "max_depth": 6, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 350, "depth": 3, "learning_rate": 0.1}

SVM_C = 0.801714
MLP_INPUT_DIM = X_full_train_np.shape[1]
MLP_HIDDEN_UNITS = (32, 48, 48)
CNN1D_CONFIG = {"nf": 64, "k": 4, "d": 80}

In [7]:
# ==========================================
# 3. CARGA DE MODELOS ENTRENADOS
# ==========================================

X_test_np_arr = np.array(X_test_np)

MODEL_PATHS = {
    "rf": os.path.join(MODEL_DIR, "rf_iot23.joblib"),
    "xgb": os.path.join(MODEL_DIR, "xgb_iot23.joblib"),
    "lgbm": os.path.join(MODEL_DIR, "lgbm_iot23.joblib"),
    "catboost": os.path.join(MODEL_DIR, "catboost_iot23.joblib"),
    "svm": os.path.join(MODEL_DIR, "svm_iot23.joblib"),
    "mlp": os.path.join(MODEL_DIR, "mlp_iot23.joblib"),
    "cnn": os.path.join(MODEL_DIR, "cnn_iot23.joblib"),
}

def load_model(model_key):
    model_path = MODEL_PATHS[model_key]
    print(f"Cargando {model_key} desde {model_path}...")
    return joblib.load(model_path)

rf_model = load_model("rf")
xgb_model = load_model("xgb")
lgbm_model = load_model("lgbm")
cat_model = load_model("catboost")

Cargando rf desde model/iot-23/rf_iot23.joblib...
Cargando xgb desde model/iot-23/xgb_iot23.joblib...
Cargando lgbm desde model/iot-23/lgbm_iot23.joblib...
Cargando catboost desde model/iot-23/catboost_iot23.joblib...


In [8]:
# ==========================================

svm_model = load_model("svm")

# ==========================================

print("\nPreparando vistas del dataset para cada familia de modelos...")
X_test_np_arr = np.array(X_test_np, dtype=np.float32)

mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train_np).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np_arr).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train_np).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np_arr).astype(np.float32)
X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)

DATASET_VIEWS = {
    "raw": {"train": X_full_train_np, "test": X_test_np_arr},
    "standard": {"train": X_train_scaled_mlp, "test": X_test_scaled_mlp},
    "minmax": {"train": X_train_scaled_cnn, "test": X_test_scaled_cnn},
}

print("Vistas disponibles:", ", ".join(DATASET_VIEWS.keys()))

# ==========================================

with tf.device(MLP_ATTACK_DEVICE):
    mlp_model = load_model("mlp")

# ==========================================

with tf.device(CNN_ATTACK_DEVICE):
    cnn_model = load_model("cnn")

Cargando svm desde model/iot-23/svm_iot23.joblib...

Preparando vistas del dataset para cada familia de modelos...


Vistas disponibles: raw, standard, minmax
Cargando mlp desde model/iot-23/mlp_iot23.joblib...


I0000 00:00:1779209129.664855 1413974 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43487 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:e1:00.0, compute capability: 8.9


Cargando cnn desde model/iot-23/cnn_iot23.joblib...


In [9]:
# Evaluamos en test MLP para tener un F1-score de referencia antes de los ataques adversarios

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM, MLP_HIDDEN_UNITS)
cnn_model_cpu = clone_keras_model_to_cpu(
    build_cnn1d_model,
    cnn_model,
    X_train_tabular_cnn.shape[1],
    CNN1D_CONFIG["nf"],
    CNN1D_CONFIG["k"],
    CNN1D_CONFIG["d"]
)
def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)
def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()

def cnn_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = cnn_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)

# F1-score de referencia en test para MLP
y_test_pred_mlp = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1 = f1_score(y_test_np, y_test_pred_mlp, pos_label=0)
print(f"\nMLP F1-score en test: {mlp_f1:.4f}")


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
I0000 00:00:1779209130.749845 1414698 service.cc:153] XLA service 0x7fc3d8646560 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779209130.749884 1414698 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779209130.755913 1414698 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779209130.827466 1414698 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



MLP F1-score en test: 0.9941


In [10]:
# Extraemos restricciones tabulares a partir del train original
feature_names = feature_columns

X_full_train_df = pl.DataFrame(X_full_train_np, schema=feature_names)

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": X_full_train_df.min().row(0),
    "max": X_full_train_df.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

clip_values_raw = (feature_mins, feature_maxs)

feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)

feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).reshape(-1, 1).astype(np.float32)

one_hot_columns = [
    col for col in feature_names
    if col.startswith("proto_") or col.startswith("conn_state_")
]

attack_mask = np.array(
    [0.0 if col in one_hot_columns else 1.0 for col in feature_names],
    dtype=np.float32
)
attack_mask_cnn = attack_mask.reshape(-1, 1).astype(np.float32)

# ART exige min < max en todas las columnas.
tiny = np.float32(1e-6)
feature_maxs_mlp = np.where(feature_maxs_mlp <= feature_mins_mlp, feature_mins_mlp + tiny, feature_maxs_mlp)
feature_maxs_cnn = np.where(feature_maxs_cnn <= feature_mins_cnn, feature_mins_cnn + tiny, feature_maxs_cnn)

clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para IOT-23:")
display(tabular_constraints_df)

print("Columnas one-hot bloqueadas:")
print(one_hot_columns)


Restricciones tabulares extraidas para IOT-23:


feature,min,max
str,f64,f64
"""id.orig_p""",3.0,65394.0
"""id.resp_p""",0.0,65535.0
"""proto_icmp""",0.0,1.0
"""proto_tcp""",0.0,1.0
"""proto_udp""",0.0,1.0
…,…,…
"""conn_state_S1""",0.0,1.0
"""conn_state_S2""",0.0,1.0
"""conn_state_SF""",0.0,1.0


Columnas one-hot bloqueadas:
['proto_icmp', 'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR']


In [11]:
# ==========================================
# FASE 1. ENVOLVER EL MODELO (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model,
    clip_values=clip_values_mlp,
    use_logits=False
)

print("Envolviendo el modelo CNN en ART con restricciones tabulares...")

clasificador_art_cnn = KerasClassifier(
    model=cnn_model,
    clip_values=clip_values_cnn,
    use_logits=False
)


Envolviendo el modelo MLP en ART con restricciones tabulares...
Envolviendo el modelo CNN en ART con restricciones tabulares...


In [12]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

EPS_VALUES = [0.01, 0.03, 0.05, 0.075, 0.1, 0.2, 0.5]

MODELOS_TABLA = ["RF", "XGB", "LGBM", "CatBoost", "SVM", "MLP", "CNN"]

modelos_clasicos = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
    "SVM": svm_model,
}

tiny_step = np.float32(1e-6)


In [13]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_fgsm_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando FGSM sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_vector_mlp = eps_vector_mlp * attack_mask
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, tiny_step).astype(np.float32)

    ataque_fgsm_mlp = FastGradientMethod(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        batch_size=ATTACK_BATCH_SIZE,
    )
    
    with tf.device(MLP_ATTACK_DEVICE):
        x_test_fgsm_mlp = ataque_fgsm_mlp.generate(
            x=x_test_mlp_attack,
            mask=attack_mask
        )

    x_test_fgsm_mlp_std = x_test_fgsm_mlp.astype(np.float32)
    x_test_fgsm_mlp_raw = mlp_scaler.inverse_transform(x_test_fgsm_mlp_std).astype(np.float32)
    x_test_fgsm_mlp_minmax = cnn_scaler.transform(x_test_fgsm_mlp_raw).astype(np.float32)
    x_test_fgsm_mlp_cnn = x_test_fgsm_mlp_minmax.reshape(
        x_test_fgsm_mlp_minmax.shape[0],
        x_test_fgsm_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_fgsm_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_fgsm_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_fgsm_mlp.append(fila_resultados)

print("Evaluacion FGSM asociada al MLP completada.")


Generando FGSM sobre MLP para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [18:45:44] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
 

Evaluacion FGSM asociada al MLP completada.


In [14]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_fgsm_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando FGSM sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_vector_cnn = eps_vector_cnn * attack_mask_cnn
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, tiny_step).astype(np.float32)

    ataque_fgsm_cnn = FastGradientMethod(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        batch_size=ATTACK_BATCH_SIZE,
    )

    with tf.device(CNN_ATTACK_DEVICE):
        x_test_fgsm_cnn = ataque_fgsm_cnn.generate(
            x=x_test_cnn_attack,
            mask=attack_mask_cnn
        )

    x_test_fgsm_cnn_cnn = x_test_fgsm_cnn.astype(np.float32)
    x_test_fgsm_cnn_minmax = x_test_fgsm_cnn_cnn.reshape(
        x_test_fgsm_cnn_cnn.shape[0],
        x_test_fgsm_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_fgsm_cnn_raw = cnn_scaler.inverse_transform(x_test_fgsm_cnn_minmax).astype(np.float32)
    x_test_fgsm_cnn_std = mlp_scaler.transform(x_test_fgsm_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_fgsm_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_fgsm_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_fgsm_cnn.append(fila_resultados)

print("Evaluacion FGSM asociada a la CNN completada.")


Generando FGSM sobre CNN para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have va

Evaluacion FGSM asociada a la CNN completada.


In [15]:
import pandas as pd

tabla_f1_fgsm_mlp = pd.DataFrame(resultados_fgsm_mlp).set_index("Limite perturbacion")
tabla_f1_fgsm_mlp = tabla_f1_fgsm_mlp[MODELOS_TABLA].round(4)

tabla_f1_fgsm_cnn = pd.DataFrame(resultados_fgsm_cnn).set_index("Limite perturbacion")
tabla_f1_fgsm_cnn = tabla_f1_fgsm_cnn[MODELOS_TABLA].round(4)

print("Tabla FGSM asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_fgsm_mlp)

print("Tabla FGSM asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_fgsm_cnn)


Tabla FGSM asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.6736,0.6092,0.5789,0.5784,0.9437,0.8052,0.9484
0.030,0.7356,0.5764,0.5790,0.5834,0.7617,0.5243,0.7623
0.050,0.7620,0.5771,0.5786,0.6219,0.2217,0.2314,0.6059
0.075,0.7619,0.5766,0.5786,0.6182,0.2203,0.1808,0.5887
0.100,0.7793,0.5768,0.5789,0.6119,0.2197,0.1786,0.6225
0.200,0.5790,0.6088,0.6097,0.5916,0.2155,0.1782,0.5888
0.500,0.5781,0.6085,0.6096,0.5916,0.2155,0.1782,0.5829


Tabla FGSM asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.6318,0.6104,0.5769,0.5786,0.8853,0.8944,0.9346
0.030,0.6167,0.5778,0.5769,0.6350,0.7888,0.8942,0.7642
0.050,0.6355,0.5777,0.5782,0.8217,0.7888,0.8958,0.5756
0.075,0.6541,0.5775,0.5782,0.8072,0.7634,0.8832,0.6223
0.100,0.6234,0.5769,0.5776,0.7083,0.7605,0.8711,0.6581
0.200,0.5826,0.6267,0.6268,0.5784,0.7602,0.8703,0.6012
0.500,0.5717,0.6194,0.6200,0.5783,0.7611,0.8588,0.5685


In [16]:
# ========================================================
# FASE 5. LANZAR ATAQUE PGD PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

PGD_MAX_ITER = 20

print("Configurando PGD para varios limites de perturbacion...")
print(f"Iteraciones PGD: {PGD_MAX_ITER}")


Configurando PGD para varios limites de perturbacion...
Iteraciones PGD: 20


In [18]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_pgd_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando PGD sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:

    print(f"PGD MLP - eps = {eps_base}")
    
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_vector_mlp = eps_vector_mlp * attack_mask
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, tiny_step).astype(np.float32)

    ataque_pgd_mlp = ProjectedGradientDescent(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        max_iter=PGD_MAX_ITER,
        batch_size=ATTACK_BATCH_SIZE,
        verbose=False
    )

    with tf.device(MLP_ATTACK_DEVICE):
        x_test_pgd_mlp = ataque_pgd_mlp.generate(
            x=x_test_mlp_attack,
            mask=attack_mask
        )

    x_test_pgd_mlp_std = x_test_pgd_mlp.astype(np.float32)
    x_test_pgd_mlp_raw = mlp_scaler.inverse_transform(x_test_pgd_mlp_std).astype(np.float32)
    x_test_pgd_mlp_minmax = cnn_scaler.transform(x_test_pgd_mlp_raw).astype(np.float32)
    x_test_pgd_mlp_cnn = x_test_pgd_mlp_minmax.reshape(
        x_test_pgd_mlp_minmax.shape[0],
        x_test_pgd_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_pgd_mlp.append(fila_resultados)

print("Evaluacion PGD asociada al MLP completada.")


Generando PGD sobre MLP para varios limites de perturbacion...
PGD MLP - eps = 0.01


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD MLP - eps = 0.03


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD MLP - eps = 0.05


...

PGD MLP - eps = 0.2


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD MLP - eps = 0.5


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Evaluacion PGD asociada al MLP completada.


In [19]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_pgd_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando PGD sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:

    print(f"PGD CNN - eps = {eps_base}")

    
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_vector_cnn = eps_vector_cnn * attack_mask_cnn
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, tiny_step).astype(np.float32)

    ataque_pgd_cnn = ProjectedGradientDescent(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        max_iter=PGD_MAX_ITER,
        batch_size=ATTACK_BATCH_SIZE,
        verbose=False
    )

    with tf.device(CNN_ATTACK_DEVICE):
        x_test_pgd_cnn = ataque_pgd_cnn.generate(
            x=x_test_cnn_attack,
            mask=attack_mask_cnn
        )

    x_test_pgd_cnn_cnn = x_test_pgd_cnn.astype(np.float32)
    x_test_pgd_cnn_minmax = x_test_pgd_cnn_cnn.reshape(
        x_test_pgd_cnn_cnn.shape[0],
        x_test_pgd_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_pgd_cnn_raw = cnn_scaler.inverse_transform(x_test_pgd_cnn_minmax).astype(np.float32)
    x_test_pgd_cnn_std = mlp_scaler.transform(x_test_pgd_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_pgd_cnn.append(fila_resultados)

print("Evaluacion PGD asociada a la CNN completada.")


Generando PGD sobre CNN para varios limites de perturbacion...
PGD CNN - eps = 0.01


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD CNN - eps = 0.03


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD CNN - eps = 0.05


...

PGD CNN - eps = 0.2


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


PGD CNN - eps = 0.5


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Evaluacion PGD asociada a la CNN completada.


In [20]:
tabla_f1_pgd_mlp = pd.DataFrame(resultados_pgd_mlp).set_index("Limite perturbacion")
tabla_f1_pgd_mlp = tabla_f1_pgd_mlp[MODELOS_TABLA].round(4)

tabla_f1_pgd_cnn = pd.DataFrame(resultados_pgd_cnn).set_index("Limite perturbacion")
tabla_f1_pgd_cnn = tabla_f1_pgd_cnn[MODELOS_TABLA].round(4)

print("Tabla PGD asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_pgd_mlp)

print("Tabla PGD asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_pgd_cnn)


Tabla PGD asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.6166,0.6158,0.5843,0.5836,0.8446,0.6041,0.9472
0.030,0.6071,0.5860,0.5871,0.5870,0.5768,0.5777,0.8210
0.050,0.6421,0.6094,0.6247,0.6321,0.5825,0.5940,0.8034
0.075,0.7416,0.8212,0.7450,0.6228,0.6148,0.6254,0.9190
0.100,0.8031,0.6553,0.6383,0.6161,0.7209,0.7010,0.8232
0.200,0.9104,0.6913,0.6927,0.6245,0.6351,0.6678,0.6924
0.500,0.8259,0.8859,0.6795,0.5916,0.4975,0.5271,0.5863


Tabla PGD asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.6316,0.6104,0.5769,0.5787,0.8723,0.8598,0.9277
0.030,0.6130,0.5782,0.5769,0.6345,0.8169,0.8785,0.7641
0.050,0.6307,0.5786,0.5770,0.9208,0.9299,0.8903,0.5785
0.075,0.6711,0.5780,0.5770,0.9088,0.8301,0.8866,0.5797
0.100,0.7219,0.5787,0.5790,0.8818,0.7742,0.5647,0.5787
0.200,0.6980,0.6433,0.6442,0.7568,0.3331,0.3438,0.5782
0.500,0.5783,0.5783,0.5783,0.5787,0.2547,0.2383,0.5783
